<a href="https://colab.research.google.com/github/hallison-dev/matching_pipeline/blob/main/02_23_matching_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# CELL 1: Install Dependencies & Load Libraries
import sys
import subprocess

def install_and_import():
    print("Step 1/3: Installing PyYAML, rank-bm25, sentence-transformers, and tqdm...")
    # Ensuring all libraries including tqdm are present in your local venv
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rank-bm25", "sentence-transformers", "numpy", "PyYAML", "tqdm"])

    print("Step 2/3: Loading ML Frameworks (this takes a moment)...")
    import json
    import numpy as np
    import re
    import yaml
    import os
    from datetime import datetime
    from rank_bm25 import BM25Okapi
    from sentence_transformers import SentenceTransformer
    from tqdm.auto import tqdm # Progress bar support

    print(f"[{datetime.now().strftime('%H:%M:%S')}] Step 3/3: ✅ All libraries loaded successfully.")
    return json, np, re, yaml, os, datetime, BM25Okapi, SentenceTransformer, tqdm

# Execute the loading
json, np, re, yaml, os, datetime, BM25Okapi, SentenceTransformer, tqdm = install_and_import()

Step 1/3: Installing PyYAML, rank-bm25, sentence-transformers, and tqdm...
Step 2/3: Loading ML Frameworks (this takes a moment)...
[05:06:32] Step 3/3: ✅ All libraries loaded successfully.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

# --- VERIFICATION BLOCK ---
try:
    from rank_bm25 import BM25Okapi
    from sentence_transformers import SentenceTransformer
    print(f"[{datetime.now().strftime('%H:%M:%S')}] ✅ Core libraries loaded successfully.")
except ImportError as e:
    print(f"❌ Critical Error: Missing dependency. {e}")
    print("Please run Cell 1 to install required libraries.")


[05:15:36] ✅ Core libraries loaded successfully.


In [ ]:
# CELL 2: Ultra-Greedy Agnostic Metadata Extractor
# -----------------------------------------------
# This version is "blind" to OpenAPI but "greedy" for tokens.
# It captures EVERY unique key and short string in the YAML tree,
# ensuring nested sub-items (fields) are captured for the match.

def ultra_greedy_extractor(data):
    """
    Recursively crawls ANY YAML structure.
    - Captures all Dictionary Keys as potential 'Fields'.
    - Captures short String Values (under 40 chars) as potential 'Fields'.
    - Captures long String Values as 'Descriptions'.
    """
    found_fields = set()
    found_text = []

    if isinstance(data, dict):
        for key, value in data.items():
            # Treat keys as technical identifiers
            # We ignore only the most basic OAI plumbing keywords to keep the search clean
            internal_plumbing = {'type', 'format', 'required', 'in', 'ref', '$ref'}
            if isinstance(key, str) and key.lower() not in internal_plumbing:
                found_fields.add(key)

            # Recurse deep into nested schemas, requestBodies, and components
            f, t = ultra_greedy_extractor(value)
            found_fields.update(f)
            found_text.extend(t)

    elif isinstance(data, list):
        for item in data:
            f, t = ultra_greedy_extractor(item)
            found_fields.update(f)
            found_text.extend(t)

    elif isinstance(data, str):
        # If it's short, it's likely a field name, enum value, or identifier
        if 1 < len(data) < 40:
            found_fields.add(data)
        # If it's long, it's likely a description or technical context
        elif len(data) >= 40:
            found_text.append(data)

    return found_fields, found_text

def extract_api_metadata(yaml_file_path):
    """Ingests YAML and flattens it using the ultra-greedy extractor."""
    try:
        with open(yaml_file_path, 'r', encoding='utf-8') as f:
            spec = yaml.safe_load(f)
    except Exception as e:
        print(f"❌ Error reading {yaml_file_path}: {e}")
        return []

    if not spec: return []

    # Process the paths
    root = spec.get('paths', {'/': spec})
    cleaned_endpoints = []

    for path, methods in root.items():
        if not isinstance(methods, dict): continue
        for method, details in methods.items():
            if not isinstance(details, dict): continue
            try:
                # Capture everything related to this specific endpoint
                fields, texts = ultra_greedy_extractor(details)

                cleaned_endpoints.append({
                    "source_file": os.path.basename(yaml_file_path),
                    "path": path,
                    "method": method.upper(),
                    "description": " ".join(texts[:25]), # Increased context window
                    "fields": list(fields),
                    "raw_source": yaml.dump({path: {method: details}})
                })
            except Exception as e:
                print(f"⚠️ Error on {method} {path}: {e}")

    return cleaned_endpoints

In [ ]:
# CELL 3: Configuration & Path Check
# ----------------------------------
# This cell is updated with your specific Google Drive paths.

import os

# Your base directory
folder_path = "/content/drive/MyDrive"

# 1. LEGACY / OLD API POOL (The "Ancestors")
# These are the original versions we are migrating FROM.
v1_filenames = [
    "jpmc_old_specification.yaml",
    "jpmc_old_account_balances_1.0.5.yaml"
]

# 2. TARGET / NEW API POOL (The "Super-Endpoints")
# This is the new consolidated version we are migrating TO.
v2_filenames = [
    "jpmc_new_account_activity_1.0.3.yaml"
]

v1_master_corpus = []
v2_target_list = []

print(f"📍 Current Working Directory: {os.getcwd()}")

# Verification check
if os.path.exists(folder_path):
    print(f"✅ Drive folder found!")
    drive_files = os.listdir(folder_path)

    # Check if our specific files exist in the list
    for f in v1_filenames + v2_filenames:
        if f in drive_files:
            print(f"   [FOUND]: {f}")
        else:
            print(f"   [MISSING]: {f} - Please verify the filename in your Google Drive.")
else:
    print(f"❌ ERROR: Cannot find folder at {folder_path}. Ensure Drive is mounted.")

📍 Current Working Directory: /content
✅ Drive folder found!
   [FOUND]: jpmc_old_specification.yaml
   [FOUND]: jpmc_old_account_balances_1.0.5.yaml
   [FOUND]: jpmc_new_account_activity_1.0.3.yaml


In [ ]:
# --- UPDATED CELL 4: Ingestion Execution ---
import json

print("--- 📥 STARTING INGESTION ---")
try:
    # Process Legacy Files
    for fname in v1_filenames:
        f_path = os.path.join(folder_path, fname)
        if not os.path.exists(f_path):
            print(f"❌ File not found in Drive: {fname}")
            continue

        results = extract_api_metadata(f_path)
        print(f"✅ Loaded: {fname:<30} | Found: {len(results)} fields")
        v1_master_corpus.extend(results)

    # Process Target Files
    for fname in v2_filenames:
        f_path = os.path.join(folder_path, fname)
        if not os.path.exists(f_path):
            print(f"❌ File not found in Drive: {fname}")
            continue

        results = extract_api_metadata(f_path)
        print(f"✅ Loaded: {fname:<30} | Found: {len(results)} fields (TARGET)")
        v2_target_list.extend(results)

    # Final summary and Preview
    if v1_master_corpus:
        print(f"\n📊 TOTALS: {len(v1_master_corpus)} Legacy fields | {len(v2_target_list)} Target fields")
        print("\n--- 🔍 PREVIEW OF FIRST RECORD ---")
        print(json.dumps(v1_master_corpus[0], indent=2))
    else:
        print("\n⚠️ Warning: No data was loaded. Please check if your YAML files contain valid metadata.")

except Exception as e:
    print(f"❌ Ingestion Failed: {e}")

--- 📥 STARTING INGESTION ---
✅ Loaded: jpmc_old_specification.yaml    | Found: 1 fields
✅ Loaded: jpmc_old_account_balances_1.0.5.yaml | Found: 1 fields
✅ Loaded: jpmc_new_account_activity_1.0.3.yaml | Found: 2 fields (TARGET)

📊 TOTALS: 2 Legacy fields | 2 Target fields

--- 🔍 PREVIEW OF FIRST RECORD ---
{
  "source_file": "jpmc_old_specification.yaml",
  "path": "/transactions",
  "method": "GET",
  "description": "Array of account number(s) you wish to retrieve the transaction\ndetails for.  Optionally used to specify the relative date type for which the\ntransactions will be retrieved. Valid values are: 'CURRENT_DAY' and\n'PRIOR_DAY'. If no dates are provided, defaults to current day. Optionally used to specify the start date of the date range for\nwhich the transactions will be retrieved. If no dates are provided,\ndefaults to current day. Optionally used to specify the end date of the date range for which\nthe transactions will be retrieved. If no dates are provided,\ndefaults to

In [ ]:
# CELL 5: Search Surface Creation (BM25 Indexing)
# ----------------------------------------------
# This cell creates the 'Search Blobs'—the text the AI actually reads.
# It then initializes the Lexical index for fast keyword matching.

def tokenize(text):
    return re.findall(r'\w+', text.lower())

def create_search_blob(obj):
    # We combine the path, description, and every single sub-item field
    # to create a high-density metadata fingerprint.
    return f"{obj['path']} {obj['description']} {' '.join(obj['fields'])}".lower()

# Build the blobs for the Legacy Master Corpus
v1_search_blobs = [create_search_blob(item) for item in v1_master_corpus]
tokenized_v1_corpus = [tokenize(blob) for blob in v1_search_blobs]

if v1_search_blobs:
    bm25 = BM25Okapi(tokenized_v1_corpus)
    print(f"✅ Lexical Index Initialized with {len(v1_search_blobs)} legacy endpoints.")
else:
    print("❌ ERROR: No search blobs created. Is v1_master_corpus empty?")

✅ Lexical Index Initialized with 2 legacy endpoints.


In [ ]:
# CELL 6: Semantic Embedding Generation (BGE Model)
# ------------------------------------------------
# This cell loads the Transformer model and creates the vector embeddings.
# This is the "heavy lifting" that allows the AI to understand that
# 'accountNumbers' (New) is related to 'accountId' (Old).

print("\n--- 🧠 LOADING SEMANTIC MODEL ---")
try:
    # We use BGE-Base, one of the most efficient models for technical documentation
    model = SentenceTransformer("BAAI/bge-base-en-v1.5")

    print("Encoding legacy corpus into vectors (this takes a moment on CPU)...")
    v1_embeddings = model.encode(v1_search_blobs, convert_to_tensor=True, show_progress_bar=True)

    print("✅ Semantic embeddings generated and ready for matching.")
except Exception as e:
    print(f"❌ Semantic Error: {e}")


--- 🧠 LOADING SEMANTIC MODEL ---


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding legacy corpus into vectors (this takes a moment on CPU)...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Semantic embeddings generated and ready for matching.


In [ ]:
# CELL 7: Matching Engine & Export
# -------------------------------
# This cell runs the RRF (Reciprocal Rank Fusion) logic.
# It uses both the Keyword index and the Semantic vectors to find the best V1 match.

def get_rrf_matches(v2_item, k_constant=60, top_n=3, threshold=0.015):
    query_blob = create_search_blob(v2_item)

    # 1. Lexical (BM25) Rank
    bm25_scores = bm25.get_scores(tokenize(query_blob))
    bm25_ranks = np.argsort(-bm25_scores)

    # 2. Semantic (BGE Vector) Rank
    query_embed = model.encode(query_blob, convert_to_tensor=True, show_progress_bar=False)

    # Ensure compatibility with NumPy dot products
    v1_arr = v1_embeddings.cpu().numpy()
    q_arr = query_embed.cpu().numpy()
    cosine_scores = np.dot(v1_arr, q_arr)
    semantic_ranks = np.argsort(-cosine_scores)

    # 3. Reciprocal Rank Fusion (Fusing both results)
    rrf_scores = np.zeros(len(v1_master_corpus))
    for rank, idx in enumerate(bm25_ranks):
        rrf_scores[idx] += 1 / (k_constant + rank + 1)
    for rank, idx in enumerate(semantic_ranks):
        rrf_scores[idx] += 1 / (k_constant + rank + 1)

    sorted_indices = np.argsort(-rrf_scores)

    # Cast scores for JSON compatibility
    top_score = float(rrf_scores[sorted_indices[0]])

    candidates = []
    for i in range(min(top_n, len(v1_master_corpus))):
        idx = sorted_indices[i]
        candidates.append({
            "v1_path": v1_master_corpus[idx]["path"],
            "v1_source": v1_master_corpus[idx]["source_file"],
            "rrf_score": float(round(rrf_scores[idx], 4)),
            "match_data": v1_master_corpus[idx]
        })

    return candidates, bool(top_score < threshold), top_score

mapping_results = []
print(f"\n{'TARGET PATH (V2)':<35} | {'LEGACY MATCH (V1)':<35} | {'SCORE'}")
print("-" * 85)

for v2_item in v2_target_list:
    candidates, is_net_new, score = get_rrf_matches(v2_item)
    match_display = candidates[0]['v1_path'] if not is_net_new else "✨ [NET NEW]"
    print(f"{v2_item['path']:<35} | {match_display:<35} | {score:.4f}")

    mapping_results.append({
        "v2_endpoint": v2_item,
        "is_net_new": is_net_new,
        "confidence_score": score,
        "v1_candidates": candidates
    })

# Save results for downstream cells (9 and 10)
with open('mapping_candidates.json', 'w') as f:
    json.dump(mapping_results, f, indent=4)

print("\n✅ Saved 'mapping_candidates.json' for Stage 3 Analysis.")


TARGET PATH (V2)                    | LEGACY MATCH (V1)                   | SCORE
-------------------------------------------------------------------------------------
/transactions                       | /transactions                       | 0.0325
/transactions/details               | /transactions                       | 0.0328

✅ Saved 'mapping_candidates.json' for Stage 3 Analysis.


In [ ]:

TARGET PATH (V2)                    | LEGACY MATCH (V1)                   | SCORE
-------------------------------------------------------------------------------------
/transactions                       | /transactions                       | 0.0325
/transactions/details               | /transactions                       | 0.0328

✅ Saved 'mapping_candidates.json' for Stage 3 Analysis.



TARGET PATH (V2)                    | HYBRID MATCH (V1)                   | SCORE
-------------------------------------------------------------------------------------


Performing Semantic Fusion:   0%|          | 0/3 [00:00<?, ?it/s]

/transactions                       | /balance                            | 0.0328
/transactions/details               | /balance                            | 0.0328
/transactions                       | /balance                            | 0.0328

✅ Hybrid Mapping Saved. Run Cell 9 for Sub-item Lineage.


In [ ]:
# CELL 8: Detailed Result Inspector (Consolidation View)
# -----------------------------------------------------
# Use this cell to verify how a "Super Endpoint" is pulling from
# multiple Legacy ancestors (e.g. Balances + Transactions).

print("--- 🔍 CONSOLIDATION ANALYSIS: V2 SUPER-ENDPOINT ANCESTRY ---")

for entry in mapping_results:
    v2_path = entry['v2_endpoint']['path']
    v2_fields = set(entry['v2_endpoint']['fields'])

    print(f"\nTARGET DESTINATION (V2): {v2_path}")
    print(f"Total Fields in V2: {len(v2_fields)}")
    print(f"Status: {'✨ [NET NEW]' if entry['is_net_new'] else '🔗 [CONSOLIDATED MATCH]'}")

    print("\nPROBABLE ANCESTORS (V1 Candidates):")
    for i, cand in enumerate(entry['v1_candidates']):
        v1_fields = set(cand['match_data']['fields'])
        # Find which fields in V2 exist in this specific V1 candidate
        overlap = v2_fields.intersection(v1_fields)

        print(f"  {i+1}. {cand['v1_path']:<30}")
        print(f"     | Score: {cand['rrf_score']:.4f}")
        print(f"     | Field Contribution: {len(overlap)} fields shared (e.g., {list(overlap)[:3]})")
        print(f"     | Source File: {cand['v1_source']}")

    print("\n" + "="*60)

print("\nNEXT STEP: Stage 3 will now use these ancestors to generate the field-level map (a1 -> s1, b1 -> s4).")

--- 🔍 CONSOLIDATION ANALYSIS: V2 SUPER-ENDPOINT ANCESTRY ---

TARGET DESTINATION (V2): /transactions
Total Fields in V2: 41
Status: 🔗 [CONSOLIDATED MATCH]

PROBABLE ANCESTORS (V1 Candidates):
  1. /transactions                 
     | Score: 0.0325
     | Field Contribution: 15 fields shared (e.g., ['parameters', '403', '503'])
     | Source File: jpmc_old_specification.yaml
  2. /balance                      
     | Score: 0.0325
     | Field Contribution: 12 fields shared (e.g., ['requestBody', 'parameters', 'tags'])
     | Source File: jpmc_old_account_balances_1.0.5.yaml


TARGET DESTINATION (V2): /transactions/details
Total Fields in V2: 38
Status: 🔗 [CONSOLIDATED MATCH]

PROBABLE ANCESTORS (V1 Candidates):
  1. /transactions                 
     | Score: 0.0328
     | Field Contribution: 15 fields shared (e.g., ['parameters', '403', '503'])
     | Source File: jpmc_old_specification.yaml
  2. /balance                      
     | Score: 0.0323
     | Field Contribution: 12 field

In [ ]:
# CELL 9: Atomic Sub-item Lineage Mapper
# -------------------------------------
# This cell breaks down the Super-Endpoint field-by-field to show
# the specific Legacy Field names that match.

print("\n--- 🔬 ATOMIC SUB-ITEM LINEAGE: FIELD-LEVEL ORIGINS ---")

for entry in mapping_results:
    v2_path = entry['v2_endpoint']['path']
    v2_fields = entry['v2_endpoint']['fields']
    candidates = entry['v1_candidates']

    print(f"\nANALYZING COMPOSITE ENDPOINT: {v2_path}")
    print(f"Total Ingredients Found in V2: {len(v2_fields)}")
    print(f"{'V2 SUB-ITEM (FIELD)':<35} | {'LEGACY (V1) SUB-ITEM':<35} | {'SOURCE'}")
    print("-" * 100)

    for v2_field in sorted(v2_fields):
        v1_match_item = "✨ [NET NEW]"
        v1_origin_path = "-"

        for cand in candidates:
            v1_fields = cand['match_data']['fields']
            v1_path = cand['v1_path']

            # Exact Match
            exact_matches = [f for f in v1_fields if v2_field.lower() == f.lower()]
            if exact_matches:
                v1_match_item = exact_matches[0]
                v1_origin_path = v1_path
                break

            # Fuzzy Match
            potential_fuzzy = [f for f in v1_fields if v2_field.lower() in f.lower() or f.lower() in v2_field.lower()]
            if potential_fuzzy:
                v1_match_item = min(potential_fuzzy, key=len)
                v1_origin_path = v1_path
                break

        print(f"{v2_field[:34]:<35} | {v1_match_item[:34]:<35} | {v1_origin_path}")

    print("\n" + "="*100)




--- 🔬 ATOMIC SUB-ITEM LINEAGE: FIELD-LEVEL ORIGINS ---

ANALYZING COMPOSITE ENDPOINT: /transactions
Total Ingredients Found in V2: 41
V2 SUB-ITEM (FIELD)                 | LEGACY (V1) SUB-ITEM                | SOURCE
----------------------------------------------------------------------------------------------------
#/components/responses/400-BadRequ  | 400                                 | /transactions
#/components/responses/401-UnAutho  | responses                           | /transactions
#/components/responses/403-Forbidd  | 403                                 | /transactions
#/components/schemas/TransactionRe  | schema                              | /transactions
200                                 | 200                                 | /transactions
400                                 | 400                                 | /transactions
401                                 | ✨ [NET NEW]                         | -
403                                 | 403                      

In [ ]:
# CELL 10: ~~Business-Level Semantic Similarity Scorer
# -------------------------------------------------
# Filters out "OAI Noise" and provides a percentage similarity
# score for business-critical fields.

print("\n--- 🎯 BUSINESS-LEVEL SEMANTIC SIMILARITY ANALYSIS ---")

TECHNICAL_NOISE = {
    '200', '400', '401', '403', 'application/json', 'content', 'description',
    'responses', 'schema', 'summary', 'tags', 'parameters', 'requestBody',
    'string', 'integer', 'boolean', 'array', 'object', 'x-examples'
}

for entry in mapping_results:
    v2_path = entry['v2_endpoint']['path']
    v2_fields = entry['v2_endpoint']['fields']
    candidates = entry['v1_candidates']

    if not candidates: continue

    print(f"\nDEEP-DIVE MAPPING: {v2_path}")
    print(f"{'V2 BUSINESS FIELD':<30} | {'BEST V1 MATCH':<25} | {'SOURCE':<10} | {'SIMILARITY'}")
    print("-" * 100)

    # Pool all fields from candidates
    v1_all_fields = []
    for cand in candidates:
        v1_all_fields.extend([(f, cand['v1_path']) for f in cand['match_data']['fields']])

    v1_names = [f[0] for f in v1_all_fields]
    v1_origins = [f[1] for f in v1_all_fields]

    v1_field_embeddings = model.encode(v1_names, convert_to_tensor=True, show_progress_bar=False)
    v1_field_arr = v1_field_embeddings.cpu().numpy()

    for v2_field in sorted(v2_fields):
        if v2_field.lower() in TECHNICAL_NOISE or any(code in v2_field for code in ['/responses/', '/schemas/']):
            continue

        v2_embed = model.encode(v2_field, convert_to_tensor=True, show_progress_bar=False)
        v2_arr = v2_embed.cpu().numpy()

        sim_scores = np.dot(v1_field_arr, v2_arr)
        best_idx = np.argmax(sim_scores)
        score = float(sim_scores[best_idx])

        status = "✅ STRONG" if score > 0.85 else "⚠️ PARTIAL" if score > 0.60 else "✨ NEW?"
        display_name = v1_names[best_idx] if status != "✨ NEW?" else "-"
        display_source = v1_origins[best_idx] if status != "✨ NEW?" else "-"
        display_score = f"{score*100:.1f}%" if status != "✨ NEW?" else "-"

        print(f"{v2_field[:29]:<30} | {display_name[:24]:<25} | {display_source[:9]:<10} | {display_score} ({status})")

print("\nANALYSIS COMPLETE.")


--- 🎯 BUSINESS-LEVEL SEMANTIC SIMILARITY ANALYSIS ---

DEEP-DIVE MAPPING: /transactions
V2 BUSINESS FIELD              | BEST V1 MATCH             | SOURCE     | SIMILARITY
----------------------------------------------------------------------------------------------------
429                            | 403                       | /transact  | 60.7% (⚠️ PARTIAL)
503                            | 503                       | /transact  | 100.0% (✅ STRONG)
Example-AccountBalance-Only-R  | Account Balances          | /balance   | 71.2% (⚠️ PARTIAL)
Example-Deafult-Report         | x-examples                | /transact  | 66.9% (⚠️ PARTIAL)
Example-Transactions-Only-Rep  | Retrieve the transaction  | /transact  | 65.5% (⚠️ PARTIAL)
Retrieve transactions and bal  | Retrieve balances         | /balance   | 91.7% (✅ STRONG)
Transactions & Balances        | Account Balances          | /balance   | 82.3% (⚠️ PARTIAL)
^[A-Za-z0-9-]{1,64}$           | 00000000000000304266256   | /balance   | 70.

In [ ]:
# CELL 11: Multi-Candidate Semantic Field Explorer
# -----------------------------------------------
# This cell shows the Top 3 Legacy candidates for every business field.
# Use this to find "Hidden Lineage" where the secondary match might be the correct one.

print("\n--- 🧭 MULTI-CANDIDATE FIELD EXPLORER (TOP 3) ---")

for entry in mapping_results:
    v2_path = entry['v2_endpoint']['path']
    v2_fields = entry['v2_endpoint']['fields']
    candidates = entry['v1_candidates']

    if not candidates: continue

    print(f"\nDEEP-DIVE (TOP 3): {v2_path}")
    print(f"{'V2 BUSINESS FIELD':<25} | {'RANK 1 (SCORE)':<28} | {'RANK 2 (SCORE)':<28} | {'RANK 3 (SCORE)':<28}")
    print("-" * 120)

    # Extract all candidate fields
    v1_all_fields = []
    for cand in candidates:
        v1_all_fields.extend([(f, cand['v1_path']) for f in cand['match_data']['fields']])

    v1_names = [f[0] for f in v1_all_fields]
    v1_origins = [f[1] for f in v1_all_fields]

    # Reuse embedding logic
    v1_field_embeddings = model.encode(v1_names, convert_to_tensor=True, show_progress_bar=False)
    v1_field_arr = v1_field_embeddings.cpu().numpy()

    for v2_field in sorted(v2_fields):
        if v2_field.lower() in TECHNICAL_NOISE or any(code in v2_field for code in ['/responses/', '/schemas/']):
            continue

        v2_embed = model.encode(v2_field, convert_to_tensor=True, show_progress_bar=False)
        v2_arr = v2_embed.cpu().numpy()

        sim_scores = np.dot(v1_field_arr, v2_arr)
        # Get indices of top 3 scores
        top_3_indices = np.argsort(-sim_scores)[:3]

        ranks = []
        for idx in top_3_indices:
            name = v1_names[idx]
            source = v1_origins[idx][:10] # Truncated for display
            score = float(sim_scores[idx])
            ranks.append(f"{name} ({source}) [{score*100:.1f}%]")

        # Ensure we always have 3 columns
        while len(ranks) < 3:
            ranks.append("-")

        print(f"{v2_field[:24]:<25} | {ranks[0]:<28} | {ranks[1]:<28} | {ranks[2]:<28}")

print("\nEXPLORER COMPLETE.")


--- 🧭 MULTI-CANDIDATE FIELD EXPLORER (TOP 3) ---

DEEP-DIVE (TOP 3): /transactions
V2 BUSINESS FIELD         | RANK 1 (SCORE)               | RANK 2 (SCORE)               | RANK 3 (SCORE)              
------------------------------------------------------------------------------------------------------------------------
429                       | 403 (/transacti) [60.7%]     | 111,222,333 (/transacti) [57.3%] | 503 (/transacti) [57.0%]    
503                       | 503 (/transacti) [100.0%]    | 2017-09-03 (/transacti) [61.3%] | 2019-09-03 (/transacti) [61.1%]
Example-AccountBalance-O  | Account Balances (/balance) [71.2%] | retrieve-balances (/balance) [64.8%] | Retrieve balances (/balance) [64.3%]
Example-Deafult-Report    | x-examples (/transacti) [66.9%] | x-examples (/balance) [66.9%] | summary (/balance) [65.0%]  
Example-Transactions-Onl  | Retrieve the transaction details (/transacti) [65.5%] | Transaction Details (/transacti) [64.5%] | Example 1 - Query by Date Range (/ba

In [ ]:
# CELL 12: High-Resolution Tabular Mapping Report
# ----------------------------------------------
# Uses Pandas to generate a perfectly aligned, interactive table.
# This is the final professional report for migration documentation.

import pandas as pd
from IPython.display import display, HTML

print("\n--- 📊 FINAL MIGRATION MAPPING TABLE ---")

all_table_data = []

for entry in mapping_results:
    v2_path = entry['v2_endpoint']['path']
    v2_fields = entry['v2_endpoint']['fields']
    candidates = entry['v1_candidates']

    if not candidates: continue

    # Pool candidate fields
    v1_all_fields = []
    for cand in candidates:
        v1_all_fields.extend([(f, cand['v1_path']) for f in cand['match_data']['fields']])

    v1_names = [f[0] for f in v1_all_fields]
    v1_origins = [f[1] for f in v1_all_fields]

    v1_field_embeddings = model.encode(v1_names, convert_to_tensor=True, show_progress_bar=False)
    v1_field_arr = v1_field_embeddings.cpu().numpy()

    for v2_field in sorted(v2_fields):
        if v2_field.lower() in TECHNICAL_NOISE or any(code in v2_field for code in ['/responses/', '/schemas/']):
            continue

        v2_embed = model.encode(v2_field, convert_to_tensor=True, show_progress_bar=False)
        v2_arr = v2_embed.cpu().numpy()

        sim_scores = np.dot(v1_field_arr, v2_arr)
        top_3_indices = np.argsort(-sim_scores)[:3]

        row = {
            "V2 Target Endpoint": v2_path,
            "V2 Field (New)": v2_field
        }

        for i, idx in enumerate(top_3_indices):
            rank = i + 1
            score = float(sim_scores[idx])
            row[f"Rank {rank}: Legacy Field"] = v1_names[idx]
            row[f"Rank {rank}: Source"] = v1_origins[idx]
            row[f"Rank {rank}: Score"] = f"{score*100:.1f}%"

        all_table_data.append(row)

# Create DataFrame
df_report = pd.DataFrame(all_table_data)

# Styling for better readability in Colab
styled_df = df_report.style.set_properties(**{
    'text-align': 'left',
    'white-space': 'nowrap',
    'font-family': 'monospace'
}).set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#f1f1f1'), ('color', '#333'), ('font-weight', 'bold')]}
])

display(styled_df)
print(f"\nTotal Business Fields Analyzed: {len(df_report)}")




--- 📊 FINAL MIGRATION MAPPING TABLE ---


,V2 Target Endpoint,V2 Field (New),Rank 1: Legacy Field,Rank 1: Source,Rank 1: Score,Rank 2: Legacy Field,Rank 2: Source,Rank 2: Score,Rank 3: Legacy Field,Rank 3: Source,Rank 3: Score
0,/transactions,429,403,/transactions,60.7%,"111,222,333",/transactions,57.3%,503,/transactions,57.0%
1,/transactions,503,503,/transactions,100.0%,2017-09-03,/transactions,61.3%,2019-09-03,/transactions,61.1%
2,/transactions,Example-AccountBalance-Only-Report,Account Balances,/balance,71.2%,retrieve-balances,/balance,64.8%,Retrieve balances,/balance,64.3%
3,/transactions,Example-Deafult-Report,x-examples,/transactions,66.9%,x-examples,/balance,66.9%,summary,/balance,65.0%
4,/transactions,Example-Transactions-Only-Report,Retrieve the transaction details,/transactions,65.5%,Transaction Details,/transactions,64.5%,Example 1 - Query by Date Range,/balance,60.6%
5,/transactions,Retrieve transactions and balances,Retrieve balances,/balance,91.7%,retrieve-balances,/balance,89.9%,Retrieve the transaction details,/transactions,85.6%
6,/transactions,Transactions & Balances,Account Balances,/balance,82.3%,Balances,/balance,78.1%,Retrieve balances,/balance,77.3%
7,/transactions,"^[A-Za-z0-9-]{1,64}$",00000000000000304266256,/balance,70.3%,description,/transactions,63.0%,description,/balance,63.0%
8,/transactions,cursor,object,/transactions,66.2%,string,/transactions,64.8%,description,/balance,63.4%
9,/transactions,default,description,/transactions,65.0%,description,/balance,65.0%,CURRENT_DAY,/balance,63.9%



Total Business Fields Analyzed: 44


In [ ]:
# CELL 13: Field-Level Migration Lineage (Deep-Dive with YAML)
# -----------------------------------------------------------
# Duplicate of Cell 12 logic but with Raw YAML snippets for Target and Rank 1.

print("\n--- 📊 FIELD-LEVEL MIGRATION LINEAGE (WITH YAML CONTEXT) ---")

detailed_mapping_rows = []

for entry in mapping_results:
    v2_ep = entry['v2_endpoint']
    v1_all_fields = []
    for cand in entry['v1_candidates']:
        v1_all_fields.extend([(f, cand['v1_path'], cand['match_data']['raw_source']) for f in cand['match_data']['fields']])

    v1_names = [f[0] for f in v1_all_fields]
    v1_origins = [f[1] for f in v1_all_fields]
    v1_yamls = [f[2] for f in v1_all_fields]

    v1_field_embeddings = model.encode(v1_names, convert_to_tensor=True, show_progress_bar=False)
    v1_field_arr = v1_field_embeddings.cpu().numpy()

    for v2_field in sorted(v2_ep['fields']):
        if v2_field.lower() in TECHNICAL_NOISE: continue
        v2_embed = model.encode(v2_field, convert_to_tensor=True, show_progress_bar=False)
        sim_scores = np.dot(v1_field_arr, v2_embed.cpu().numpy())
        top_3 = np.argsort(-sim_scores)[:3]

        row = {
            "V2 Endpoint": v2_ep['path'],
            "V2 Field": v2_field,
            "Rank 1: Legacy": v1_names[top_3[0]],
            "Rank 1: Source": v1_origins[top_3[0]],
            "Rank 1: Score": f"{sim_scores[top_3[0]]*100:.1f}%",
            "Rank 2: Legacy": v1_names[top_3[1]],
            "Rank 2: Score": f"{sim_scores[top_3[1]]*100:.1f}%",
            "Rank 3: Legacy": v1_names[top_3[2]],
            "Rank 3: Score": f"{sim_scores[top_3[2]]*100:.1f}%",
            "V2 Snippet": v2_ep['raw_source'],
            "V1 Snippet": v1_yamls[top_3[0]]
        }
        detailed_mapping_rows.append(row)

df_deep_dive = pd.DataFrame(detailed_mapping_rows)

def format_yaml(val):
    return f'<pre style="text-align:left; font-size:10px; background:#f8f9fa; padding:8px; border:1px solid #ddd; border-radius:4px; max-height:250px; overflow-y:auto;">{val}</pre>'

html_table = df_deep_dive.to_html(
    escape=False,
    index=False,
    formatters={'V2 Snippet': format_yaml, 'V1 Snippet': format_yaml}
)

display(HTML(f"""
<div style="overflow-x: auto; width: 100%; border: 1px solid #ddd; border-radius: 8px; background: white;">
    <style>
        .dataframe {{ width: 100%; min-width: 3200px; border-collapse: collapse; font-family: -apple-system, sans-serif; table-layout: fixed; }}
        .dataframe th {{ background: #2c3e50; color: white; padding: 15px; text-align: left; font-size: 13px; position: sticky; top: 0; }}
        .dataframe td {{ padding: 12px; border-bottom: 1px solid #eee; vertical-align: top; font-size: 12px; word-wrap: break-word; }}
        .dataframe tr:hover {{ background: #f1f3f5; }}
    </style>
    {html_table}
</div>
"""))



--- 📊 FIELD-LEVEL MIGRATION LINEAGE (WITH YAML CONTEXT) ---


In [ ]:
# CELL 14: Atomic Field-Level AI Payload Export (n8n Ready)
# --------------------------------------------------------
# This cell generates a JSON where EVERY ROW in the table above is its own object.
# This provides the AI with "Atomic Tasks" rather than whole endpoints.

from google.colab import files
from datetime import datetime

print("\n--- 📦 GENERATING ATOMIC FIELD-LEVEL AI PAYLOAD ---")

atomic_tasks = []

# Using the data generated for the table in Cell 13
for row in field_mapping_rows:
    task = {
        "v2_context": {
            "endpoint": row["Target Endpoint"],
            "field": row["V2 Field (New)"],
            "raw_yaml": row["V2 Snippet"]
        },
        "v1_match_candidate": {
            "field": row["V1 Field (Old)"],
            "source_endpoint": row["V1 Source"],
            "similarity_score": row["Similarity"],
            "raw_yaml": row["V1 Snippet"]
        },
        "instruction": "Analyze the semantic and technical relationship between the V2 field and the V1 candidate. Provide transformation logic."
    }
    atomic_tasks.append(task)

# Save and Trigger Download
export_filename = f"atomic_field_mappings_{datetime.now().strftime('%Y%m%d_%H%M')}.json"
with open(export_filename, 'w') as f:
    json.dump(atomic_tasks, f, indent=2)

print(f"✅ Created {len(atomic_tasks)} atomic field-mapping tasks.")
print(f"🚀 Triggering download for: {export_filename}")
files.download(export_filename)


--- 📦 GENERATING ATOMIC FIELD-LEVEL AI PAYLOAD ---
✅ Created 44 atomic field-mapping tasks.
🚀 Triggering download for: atomic_field_mappings_20260223_0645.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# CELL 14: Atomic Multi-Candidate AI Payload (n8n Ready)
# ----------------------------------------------------
# This generates a JSON where each V2 field is an atomic task with the TOP 3 V1 matches.

from google.colab import files
from datetime import datetime

print("\n--- 📦 GENERATING ATOMIC MULTI-CANDIDATE AI PAYLOAD ---")

atomic_payload = []

# Using the granular data gathered in Cell 13
for entry in mapping_results:
    v2_ep = entry['v2_endpoint']

    # Pool all fields from candidates for this specific endpoint context
    v1_pool = []
    for cand in entry['v1_candidates']:
        v1_pool.extend([(f, cand['v1_path'], cand['match_data']['raw_source']) for f in cand['match_data']['fields']])

    v1_names = [f[0] for f in v1_pool]
    v1_origins = [f[1] for f in v1_pool]
    v1_yamls = [f[2] for f in v1_pool]

    v1_embeddings_local = model.encode(v1_names, convert_to_tensor=True, show_progress_bar=False)
    v1_arr_local = v1_embeddings_local.cpu().numpy()

    for v2_field in v2_ep['fields']:
        if v2_field.lower() in TECHNICAL_NOISE: continue

        v2_embed = model.encode(v2_field, convert_to_tensor=True, show_progress_bar=False)
        sim_scores = np.dot(v1_arr_local, v2_embed.cpu().numpy())
        top_3_indices = np.argsort(-sim_scores)[:3]

        task = {
            "v2_context": {
                "endpoint": v2_ep['path'],
                "field": v2_field,
                "raw_yaml": v2_ep['raw_source']
            },
            "v1_candidates": []
        }

        for idx in top_3_indices:
            task["v1_candidates"].append({
                "field": v1_names[idx],
                "source_endpoint": v1_origins[idx],
                "similarity_score": f"{sim_scores[idx]*100:.1f}%",
                "raw_yaml": v1_yamls[idx]
            })

        atomic_payload.append(task)

export_name = f"atomic_multi_candidate_payload_{datetime.now().strftime('%Y%m%d_%H%M')}.json"
with open(export_name, 'w') as f:
    json.dump(atomic_payload, f, indent=2)

print(f"✅ Created {len(atomic_payload)} atomic tasks with 3 candidates each.")
files.download(export_name)


--- 📦 GENERATING ATOMIC MULTI-CANDIDATE AI PAYLOAD ---
✅ Created 51 atomic tasks with 3 candidates each.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# CELL 15: Atomic Multi-Candidate AI Payload (n8n Ready)
# ----------------------------------------------------
# This is the "AI-Friendly" Gold Standard. One object per field,
# providing the TOP 3 legacy candidates with full YAML context.

print("\n--- 📦 GENERATING ATOMIC MULTI-CANDIDATE AI PAYLOAD ---")

atomic_multi_payload = []

for entry in mapping_results:
    v2_ep = entry['v2_endpoint']

    # Pool all legacy fields from the top ancestors identified earlier
    v1_pool = []
    for cand in entry['v1_candidates']:
        v1_pool.extend([(f, cand['v1_path'], cand['match_data']['raw_source']) for f in cand['match_data']['fields']])

    v1_names = [f[0] for f in v1_pool]
    v1_origins = [f[1] for f in v1_pool]
    v1_yamls = [f[2] for f in v1_pool]

    v1_embeddings_local = model.encode(v1_names, convert_to_tensor=True, show_progress_bar=False)
    v1_arr_local = v1_embeddings_local.cpu().numpy()

    for v2_field in v2_ep['fields']:
        if v2_field.lower() in TECHNICAL_NOISE: continue

        v2_embed = model.encode(v2_field, convert_to_tensor=True, show_progress_bar=False)
        sim_scores = np.dot(v1_arr_local, v2_embed.cpu().numpy())
        top_3_indices = np.argsort(-sim_scores)[:3]

        task = {
            "v2_context": {
                "endpoint": v2_ep['path'],
                "field": v2_field,
                "raw_yaml": v2_ep['raw_source']
            },
            "v1_candidates": []
        }

        for idx in top_3_indices:
            task["v1_candidates"].append({
                "field": v1_names[idx],
                "source_endpoint": v1_origins[idx],
                "similarity_score": f"{sim_scores[idx]*100:.1f}%",
                "raw_yaml": v1_yamls[idx]
            })

        atomic_multi_payload.append(task)

export_name = f"atomic_multi_candidate_payload_{datetime.now().strftime('%Y%m%d_%H%M')}.json"
with open(export_name, 'w') as f:
    json.dump(atomic_multi_payload, f, indent=2)

print(f"✅ Created {len(atomic_multi_payload)} atomic tasks with 3 candidates each.")
files.download(export_name)


--- 📦 GENERATING ATOMIC MULTI-CANDIDATE AI PAYLOAD ---
✅ Created 51 atomic tasks with 3 candidates each.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>